[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/notebooks/r2/r2-q2/04_categorization.ipynb)

# R2-Q2 Week 4 — Apply the taxonomy and report the failure-mode distribution

This notebook applies the four numeric predicates committed in Week 1 to the Grad-CAM heatmaps produced in Week 3, assigning each misclassification to one of five categories — symptom-attended-but-wrong-class, background-attended, lighting-attended, leaf-shape-attended, or other/ambiguous — following the first-match-wins scheme with multi-fires routing to "other." The output is the failure-mode distribution across the misclassification set, reported at the overall taxonomy level rather than per-class, and the methods-section text documenting how each predicate was operationalized.

By the end of this notebook you will have:

- A `categorizations.parquet` containing one row per misclassification with columns recording which predicates fired, the assigned category, and the derived quantities (m_out, m_perimeter, lighting_correlation, m_in, concentration) that drove the assignment — so a reader can audit any individual categorization.
- A `taxonomy_distribution.json` recording the count and fraction of misclassifications in each of the five categories, the conjunction-handling statistics (how many images fired multiple predicates and routed to "other"), and the headline interpretation sentence describing what the failure-mode distribution looks like.
- A `taxonomy_distribution.png` bar chart showing the five-category distribution with the "other" bucket visually distinguished — so a reader can see at a glance whether failures clustered into recognizable patterns or whether the taxonomy itself needs revision.

## Before you run anything: switch to a GPU runtime

This notebook uses a large vision model (the Segment Anything Model,
or SAM) to separate the leaf from the background in each of the 81
misclassified images. SAM runs much faster on a GPU than on a CPU,
and 81 images on CPU adds up to a long wait. Colab's default runtime
is CPU-only, so you'll need to switch.

To switch:

1. In Colab's menu bar: **Runtime → Change runtime type**.
2. Under "Hardware accelerator," choose **T4 GPU**.
3. Click **Save**.

Wait for the session to come up on the new runtime (a few seconds).
Then run the cells below in order.

**If Colab tells you "no GPUs available right now":** free-tier GPU
access is shared and occasionally unavailable. Wait a few minutes
and try again. If it keeps refusing, tell your mentor — the
upgraded Colab tiers have more reliable GPU access.

In [ ]:
try:
    import irilab2026 as iri
    print(f"OK — irilab2026 {iri.__version__} is installed and all deps importable.")
    print("    → use Pattern 2 in the following Cell:")
    print("      !pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main")
except ModuleNotFoundError as e:
    print(f"MISSING — {e.name} not importable.")
    print("    → use Pattern 1 in the following Cell:")
    print("      !pip install git+https://github.com/geraldmc/irilab2026.git@main")

In [ ]:
# Pattern 1 — fresh or partial runtime (installs deps that aren't present yet)
# This skips anything pip already sees as installed. This is ideal for a new environment or when you want
# to be sure everything is up to date.

# !pip install git+https://github.com/geraldmc/irilab2026.git@main

# Pattern 2 — populated runtime (refreshes irilab2026 only, leaves deps alone)
# This is ideal for when you already have the dependencies installed and just want to update irilab2026.

# !pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main

### Confirm the GPU is in place

Before going further, confirm Colab attached a GPU. The cell below
prints `GPU detected:` followed by `True` or `False`. If it's `True`,
you're good. If it's `False`, go back to the top of the notebook and
re-do the runtime switch — the next cells will not work without a GPU.

The check is a separate cell from the main setup (below) because it's
fast and recoverable: if the GPU isn't there, you see the problem
right away with no Drive authorization prompt to dismiss first.

In [ ]:
import irilab2026 as iri

gpu_ok = iri.has_gpu()
print(f"GPU detected: {gpu_ok}")

assert gpu_ok, (
    "No GPU detected. Go to the top of the notebook and follow the "
    "runtime-switch instructions, then re-run from Setup Step 1."
)

In [ ]:
# Mount Drive and set up irilab2026 with a GPU.
import irilab2026 as iri
iri.setup(gpu_required=True)

OUTPUT_DIR = iri.output_dir("r2_q2")
print(f"Output directory: {OUTPUT_DIR}")